## 简历冲浪助手（Resume Surfer）

面向招聘场景的 **RAG 聊天助手**：把候选人 PDF 简历切块入库，再用本地 **Ollama**（`ChatOllama`）基于检索到的简历片段回答客户关于候选人的问题。

### 和本课第 5 周的关系

- PDF → Document（`PyPDFLoader`）
- 切块 + HuggingFace 嵌入 + Chroma
- Retriever + System Prompt + Gradio `ChatInterface`

### 怎么跑

1. 安装依赖（下一格 `%pip install`），并确保本机 Ollama 已拉取 `gpt-oss:20b`
2. 准备 `.env`（本格用 `load_dotenv`；密钥按你环境需要配置）
3. 运行到输入路径格时，填入简历 PDF 的本地路径
4. 建库后启动 Gradio，用自然语言问简历内容


In [ ]:
# 安装 LangChain 的 Ollama 集成包（-q 安静模式）；逻辑字符串保持原样
%pip install -q langchain-ollama


In [ ]:
# ========== 导入：Gradio UI + OpenAI/tiktoken + LangChain RAG 组件 ==========

# 标准库 os：判断向量库目录是否已存在
import os


# Gradio：聊天界面
import gradio as gr
# OpenAI 客户端类（本笔记本后续以 Ollama 为主，仍保留导入）
from openai import OpenAI

# tiktoken：按模型规则估算 token 数
import tiktoken
# load_dotenv：加载 .env 环境变量
from dotenv import load_dotenv
# ChatOllama：通过 LangChain 调本地 Ollama 对话模型
from langchain_ollama import ChatOllama
# Chroma：向量库
from langchain_chroma import Chroma
# HuggingFace 嵌入：本机计算句向量
from langchain_huggingface import HuggingFaceEmbeddings
# 消息类型：System / Human
from langchain_core.messages import SystemMessage, HumanMessage
# PDF 加载器：把简历页变成 Document 列表
from langchain_community.document_loaders import PyPDFLoader
# 递归字符切块器
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [ ]:
# ========== 常数：模型名与向量库目录 ==========

# 加载 .env（override=True：覆盖进程里已有同名变量）
load_dotenv(override=True)

# 用于 tiktoken 编码选型的 OpenAI 模型名（字符串保持原样）
MODEL_OPENAI = "gpt-4o-mini"
# 本地 Ollama 模型名（需事先 pull；字符串保持原样）
LOCAL_MODEL_OPENAI = "gpt-oss:20b"


# Chroma 持久化目录名
db_name = "resume_db"


In [ ]:
# ========== 读 PDF：只接受 .pdf，否则返回 None ==========

def read_pdf_file(file_path):
    # 扩展名检查：非 pdf 直接视为无效
    if file_path.endswith(".pdf"):
        # PyPDFLoader：按页加载为 Document
        loader = PyPDFLoader(file_path)

        # load()：真正读文件
        resume = loader.load()

        return resume
    else:
        # 非 PDF：返回 None，后面 load_to_store 会据此报错
        return None


In [ ]:
# ========== 交互输入简历路径，并加载为 Document 列表 ==========

# input 提示文案保持英文原样（影响交互显示）
resume_path = input("Enter the path to the resume file: ")
# 调用上一格函数；成功则为 List[Document]，失败为 None
resume = read_pdf_file(resume_path)


In [ ]:
# ========== HR 助手 System Prompt 模板（{context} 稍后填入检索片段）==========
# 英文 prompt 正文不翻译，以免改变模型行为

SYSTEM_PROMPT_TEMPLATE = """
You are an HR assistant, friendly assistant representing the recruiting firm.
You're chatting with a client and asked about a candidate
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""


In [ ]:
# ========== Token 估算：用 tiktoken 看简历大概有多长 ==========

# 按 MODEL_OPENAI 的编码规则取 encoder
encoding = tiktoken.encoding_for_model(MODEL_OPENAI)

# 把各页 page_content 拼成一整段纯文本
resume_text = "\n".join(doc.page_content for doc in resume)

# encode → token 列表；len 即 token 数
tokens = encoding.encode(resume_text)
token_count = len(tokens)
# 打印带千分位的总数，便于判断是否需要切块/截断
print(f"Total tokens for {MODEL_OPENAI}: {token_count:,}")


In [ ]:
# ========== 切块 → 嵌入 →（若有旧库则删除）→ 写入 Chroma ==========

def load_to_store():
    # 简历未加载成功时立刻失败，避免后面空指针
    if resume is None:
        raise ValueError("Resume not loaded")

    # 切块：500 字符，重叠 100
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )
    
    # 对简历 Document 列表切块
    chunks = text_splitter.split_documents(resume)

    # 本地句向量模型（首次会下载权重）
    embeddings = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2"
    )

    # 若目录已存在，删除旧 collection，保证本次重建干净
    if os.path.exists(db_name):
        Chroma(
            persist_directory=db_name,
            embedding_function=embeddings
        ).delete_collection()

    # 从 chunks 建库并持久化
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=db_name
    )

    # 打印入库条数
    print(f"Vectorstore created with {vectorstore._collection.count()} documents")

    return vectorstore



In [ ]:
# ========== 建库并窥探向量：条数与维度 ==========

# 执行入库，得到 vectorstore 句柄
vectorstore = load_to_store()

# 底层 collection（Chroma 原生对象）
collection = vectorstore._collection
# 当前向量条数
count = collection.count()

# 取 1 条 embedding，看向量维度（如 384）
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")


In [ ]:
# ========== Retriever + 本地 LLM + RAG 答题函数 ==========

# 默认检索器
retriever = vectorstore.as_retriever()
# ChatOllama：temperature=0，回答更稳
llm = ChatOllama(model=LOCAL_MODEL_OPENAI, temperature=0)

def answer_question(question: str, history):
    # history 由 Gradio 传入；此处未拼接多轮历史
    # 检索与问题相关的简历片段
    docs = retriever.invoke(question)
    # 拼 context
    context = "\n\n".join(doc.page_content for doc in docs)
    # 填入 system prompt
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    # System + Human 消息调用本地模型
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    # 返回纯文本给聊天界面
    return response.content


In [ ]:
# ========== 启动 Gradio ChatInterface ==========
gr.ChatInterface(answer_question).launch()
